# Fake Job Detection - FULL Model on Colab GPU

**Yeh exact same model hai jo aapne CPU pe chalaya tha!**

## Step 1: Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Upload fake_job_postings.csv to Google Drive first
!cp /content/drive/MyDrive/fake_job_postings.csv /content/

In [ ]:
!pip install transformers torch scikit-learn pandas numpy tqdm -q

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("CUDA:", torch.version.cuda)

## Step 2: Full Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('fake_job_postings.csv')
print(f"Loaded {len(df)} records")
print(df['fraudulent'].value_counts())

In [ ]:
# Text cleaning
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    return text.strip().lower()

# Prepare text
text_data = []
for _, row in df.iterrows():
    combined = f"Title: {row['title']}. "
    combined += f"Description: {clean_text(row['description'])}. "
    combined += f"Requirements: {clean_text(row['requirements'])}. "
    combined += f"Company: {clean_text(row['company_profile'])}. "
    combined += f"Benefits: {clean_text(row['benefits'])}"
    text_data.append(combined)

print(f"Prepared {len(text_data)} texts")

In [ ]:
# Metadata features
features = pd.DataFrame()

features['desc_length'] = df['description'].fillna('').apply(len)
features['title_length'] = df['title'].fillna('').apply(len)
features['req_length'] = df['requirements'].fillna('').apply(len)
features['company_profile_length'] = df['company_profile'].fillna('').apply(len)

features['has_salary'] = df['salary_range'].notna().astype(int)
features['has_department'] = df['department'].notna().astype(int)
features['has_company_logo'] = df['has_company_logo'].fillna(0).astype(int)
features['has_questions'] = df['has_questions'].fillna(0).astype(int)
features['telecommuting'] = df['telecommuting'].fillna(0).astype(int)

features['has_location'] = df['location'].notna().astype(int)
features['is_us_location'] = df['location'].fillna('').str.contains('US', case=False).astype(int)

emp_type_dummies = pd.get_dummies(df['employment_type'], prefix='emp_type', dummy_na=True)
features = pd.concat([features, emp_type_dummies], axis=1)

exp_mapping = {
    'Not Applicable': 0, 'Entry level': 1, 'Associate': 2,
    'Mid-Senior level': 3, 'Director': 4, 'Executive': 5
}
features['experience_level'] = df['required_experience'].map(exp_mapping).fillna(0)

edu_mapping = {
    'Unspecified': 0, 'High School or equivalent': 1,
    'Vocational': 2, 'Some College Coursework Completed': 3,
    'Associate Degree': 4, "Bachelor's Degree": 5,
    "Master's Degree": 6, 'Doctorate': 7, 'Professional': 8
}
features['education_level'] = df['required_education'].map(edu_mapping).fillna(0)

top_industries = df['industry'].value_counts().head(10).index
for ind in top_industries:
    features[f'industry_{ind}'] = (df['industry'] == ind).astype(int)

top_functions = df['function'].value_counts().head(10).index
for func in top_functions:
    features[f'function_{func}'] = (df['function'] == func).astype(int)

features['desc_missing'] = df['description'].isna().astype(int)
features['requirements_missing'] = df['requirements'].isna().astype(int)

print(f"Total features: {len(features.columns)}")

In [ ]:
# Scale and split
scaler = StandardScaler()
meta_scaled = scaler.fit_transform(features)
labels = df['fraudulent'].values

# Temporal split
df_sorted = df.sort_values('job_id')
sorted_indices = df_sorted.index.values

n = len(sorted_indices)
train_idx = sorted_indices[:int(0.7 * n)]
val_idx = sorted_indices[int(0.7 * n):int(0.85 * n)]

train_texts = [text_data[i] for i in train_idx]
val_texts = [text_data[i] for i in val_idx]
train_meta = meta_scaled[train_idx]
val_meta = meta_scaled[val_idx]
train_labels = labels[train_idx]
val_labels = labels[val_idx]

print(f"Train: {len(train_labels)}, Val: {len(val_labels)}")

## Step 3: Dataset

In [ ]:
from transformers import DistilBertTokenizer
from torch.utils.data import Dataset
import torch

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class FakeJobDataset(Dataset):
    def __init__(self, texts, metadata, labels, tokenizer, max_length=512):
        self.texts = texts
        self.metadata = torch.tensor(metadata, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.tokenizer = tokenizer  # Store tokenizer!
        self.max_length = max_length
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        meta = self.metadata[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'metadata': meta,
            'label': label
        }

# Create datasets - PASS TOKENIZER!
train_dataset = FakeJobDataset(train_texts, train_meta, train_labels, tokenizer)
val_dataset = FakeJobDataset(val_texts, val_meta, val_labels, tokenizer)

print(f"Datasets created!")

## Step 4: Model

In [ ]:
import torch.nn as nn
from transformers import DistilBertModel

class FakeJobDetector(nn.Module):
    def __init__(self, num_meta_features, dropout_rate=0.4):
        super().__init__()
        
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.text_dropout = nn.Dropout(0.2)
        
        for param in self.distilbert.transformer.layer[:4].parameters():
            param.requires_grad = False
        
        self.metadata_encoder = nn.Sequential(
            nn.Linear(num_meta_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        combined_dim = 768 + 64
        
        self.fusion = nn.Sequential(
            nn.Linear(combined_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate - 0.1),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
    
    def forward(self, input_ids, attention_mask, metadata):
        bert_output = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        text_features = bert_output.last_hidden_state[:, 0, :]
        text_features = self.text_dropout(text_features)
        
        meta_features = self.metadata_encoder(metadata)
        
        combined = torch.cat([text_features, meta_features], dim=1)
        logits = self.fusion(combined)
        return logits.squeeze(-1)

## Step 5: Training

In [ ]:
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, f1_score
import torch.optim as optim
from tqdm import tqdm
import numpy as np

BATCH_SIZE = 16
EPOCHS = 10
device = torch.device('cuda')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

model = FakeJobDetector(num_meta_features=train_meta.shape[1]).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {total_params:,}, Trainable: {trainable_params:,}")

# Optimizer
bert_params = list(model.distilbert.named_parameters())
other_params = list(model.metadata_encoder.parameters()) + list(model.fusion.parameters())

optimizer = optim.AdamW([
    {'params': [p for n, p in bert_params], 'lr': 2e-5},
    {'params': other_params, 'lr': 2e-4}
])

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([10.0]).to(device))

best_auc = 0
patience = 0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
    # Train
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        metadata = batch['metadata'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask, metadata)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    
    # Val
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            metadata = batch['metadata'].to(device)
            logits = model(input_ids, attention_mask, metadata)
            probs = torch.sigmoid(logits)
            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(batch['label'].numpy())
    
    auc = roc_auc_score(all_labels, all_preds)
    print(f"Loss: {train_loss/len(train_loader):.4f}, AUC: {auc:.4f}")
    
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), 'best_model_full.pt')
        print(f"✓ Saved!")
        patience = 0
    else:
        patience += 1
        if patience >= 3:
            print("Early stop")
            break

print(f"\nBest AUC: {best_auc:.4f}")

## Step 6: Download

In [ ]:
from google.colab import files
files.download('best_model_full.pt')